# Adsorption Energy of Metal Adatoms on Graphene

> **K. T. Chan, J. B. Neaton, and M. L. Cohen**
> "First-principles study of metal adatom adsorption on graphene"
> Phys. Rev. B 77, 235430 (2008)
> [DOI: 10.1103/PhysRevB.77.235430](https://doi.org/10.1103/PhysRevB.77.235430)

Computes the adsorption energy of a metal adatom on the 4x4 graphene cells created in the
[structure notebook](defect_surface_adatom_graphene.ipynb), with the manuscript's Eq. 1, ΔE = E_ag − E_a − E_g, printed
positive for a bound adatom as in Table I, together with the adatom height, the nearest adatom–C distance and the total
magnetization, and compares them with Tables I and II.

<h2 style="color:green">Usage</h2>

1. Create the materials in the [structure notebook](defect_surface_adatom_graphene.ipynb), which saves `Graphene 4x4` and the adatom cells, such as `Li on graphene 4x4 hollow`, to the `uploads` folder.
1. Set the material names and parameters in cells 1.2-1.4 (or use the defaults); each relaxed cell is saved as `<name> relaxed`.
1. Click "Run" > "Run All" to run all cells.
1. Wait for the jobs to complete.
1. Scroll down to view the results.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure the material names, model and compute parameters.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then select account and project.
1. Load the clean layer and the adatom cells by name, build one isolated atom per adatom element in a cubic box, print provenance, then save all of them to the platform, each adatom free to move along z only.
1. Configure the shared DFT model: one model for every workflow below.
1. Configure compute: get the list of clusters and create a compute configuration.
1. Relax the clean layer and the adatom cells at fixed cell with the dipole correction, reusing a relaxation job that already exists, and read the total magnetization of each adatom cell.
1. Run one total energy job per isolated atom, and read the cell energies from the relaxation jobs.
1. Retrieve the results: the adsorption energies, the adatom heights and the nearest adatom–C distances.
1. Compare with Chan, Neaton & Cohen (2008).

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|specific_examples|api_examples")

### 1.2. Materials

In [ ]:
# Names saved by defect_surface_adatom_graphene.ipynb; uncomment a line to add that adatom cell
CLEAN_LAYER_NAME = "Graphene 4x4"
ADATOM_CELL_NAMES = [
    "Li on graphene 4x4 hollow",
    # "Li on graphene 4x4 bridge",
    # "Li on graphene 4x4 top",
    # "Na on graphene 4x4 hollow",
    # "K on graphene 4x4 hollow",
    # "Sn on graphene 4x4 top",
    # "Pd on graphene 4x4 bridge",
]
ATOM_NAME = "{element} atom"
ATOM_BOX = 14.0  # Å, edge of the cubic box around each isolated atom, as in the manuscript

### 1.3. Parameters

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

ORGANIZATION_NAME = None  # set to your organization name (full or partial); otherwise, your default one is used
FOLDER = "./uploads"

TOTAL_ENERGY_SEARCH_TERM = "total_energy.json"
RELAX_WORKFLOW_SEARCH_TERM = "fixed_cell_relaxation.json"
APPLICATION_NAME = "espresso"

CLUSTER_NAME = "001"  # specify full or partial name i.e. "cluster-001" to select
QUEUE_NAME = QueueName.OF
PPN = 60  # OF on cluster-001 allows at most 60 cores per node
TIME_LIMIT = "12:00:00"  # the Li-hollow relaxation (33 atoms) converged in 7 SCF cycles, about 5 h on 36 cores

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 60  # seconds

# Chan 2008, Table I: ΔE and E_max − ΔE in eV, h and d_AC in Å; Table II: total magnetization μ_AG in μB, favored sites
MANUSCRIPT_VALUES_BY_CELL_NAME = {
    "Li on graphene 4x4 hollow": {"adsorption_energy": 1.096, "height": 1.71, "adatom_carbon_distance": 2.23,
                                  "total_magnetization": 0.00},
    "Li on graphene 4x4 bridge": {"adsorption_energy": 0.773, "energy_below_favored_site": 0.322, "height": 1.88,
                                  "adatom_carbon_distance": 2.09},
    "Li on graphene 4x4 top": {"adsorption_energy": 0.754, "energy_below_favored_site": 0.342, "height": 1.89,
                               "adatom_carbon_distance": 2.02},
    "Na on graphene 4x4 hollow": {"adsorption_energy": 0.462, "height": 2.28, "adatom_carbon_distance": 2.70,
                                  "total_magnetization": 0.27},
    "K on graphene 4x4 hollow": {"adsorption_energy": 0.802, "height": 2.60, "adatom_carbon_distance": 2.99,
                                 "total_magnetization": 0.17},
    "Sn on graphene 4x4 top": {"adsorption_energy": 0.256, "height": 2.82, "adatom_carbon_distance": 2.75,
                               "total_magnetization": 1.81},
    "Pd on graphene 4x4 bridge": {"adsorption_energy": 1.081, "height": 2.21, "adatom_carbon_distance": 2.18,
                                  "total_magnetization": 0.00},
}

### 1.4. DFT model parameters

In [ ]:
# manuscript: VASP PAW at 500 eV; ours GBRV ultrasoft at its own converged 40/200 Ry
FUNCTIONAL = "pbe"
PSEUDOPOTENTIAL_TYPE = "us"
ECUTWFC = 40   # Ry
ECUTRHO = 200  # Ry

SLAB_KGRID = [9, 9, 1]  # manuscript: 9x9x1 Γ-centered for the clean layer and the adatom cells
GAMMA_KGRID = [1, 1, 1]  # manuscript: Γ only for the isolated atoms
# manuscript: Gaussian, σ = 0.05 eV = 0.003675 Ry
SMEARING_SETTINGS = {"occupations": "smearing", "smearing": "gaussian", "degauss": 0.003675}
# manuscript: dipole correction along z; eamp = 0 keeps the sawtooth a correction, no external field
DIPOLE_SETTINGS = {"control": {"tefield": True, "dipfield": True}, "system": {"edir": 3, "eamp": 0.0, "eopreg": 0.05}}
# manuscript: forces below 0.05 eV/Å = 1.9e-3 Ry/Bohr, per force component;
# section 7.2 prints the force norm (all atoms)
RELAXATION_SETTINGS = {"forc_conv_thr": 1.9e-3, "nstep": 100}
# manuscript: the adatom relaxes along z only, the C atoms in all directions
ADATOM_CONSTRAINT = [False, False, True]
CARBON_CONSTRAINT = [True, True, True]
# manuscript: spin-polarized; the clean layer is non-magnetic and runs without spin.
# C is species 1 in the input and the adatom species 2, which starts with a magnetic moment.
ADATOM_SPIN_SETTINGS = {"nspin": 2, "starting_magnetization(2)": 0.5}
# Table II μ_A: total magnetization of each isolated atom in μB; Pd (μ_A = 0) runs without spin
ATOM_SPIN_SETTINGS_BY_ELEMENT = {
    "Li": {"nspin": 2, "tot_magnetization": 1},
    "Na": {"nspin": 2, "tot_magnetization": 1},
    "K": {"nspin": 2, "tot_magnetization": 1},
    "Sn": {"nspin": 2, "tot_magnetization": 2},
    "Pd": {},
}
MODEL_TAG = f"{FUNCTIONAL}-{PSEUDOPOTENTIAL_TYPE} {ECUTWFC}-{ECUTRHO}Ry g{SMEARING_SETTINGS['degauss']}"

SCF_UNIT = "pw_scf"
RELAX_UNIT = "pw_relax"
RELAX_OUTPUT_FILE = "pw_relax.out"

## 2. Authenticate and initialize API client
### 2.1. Authenticate

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Load the materials
### 3.1. Load the cells from the uploads folder or the platform, build the isolated atoms, and print provenance

In [ ]:
from collections import Counter
from ase import Atoms
from mat3ra.made.material import Material
from mat3ra.made.tools.convert import from_ase
from mat3ra.notebooks_utils.core.entity.material.api import load_material

slabs = {name: load_material(client, FOLDER, name, ACCOUNT_ID) for name in (CLEAN_LAYER_NAME, *ADATOM_CELL_NAMES)}
adatom_element_by_cell_name = {name: next(element for element in slabs[name].basis.elements.values if element != "C")
                               for name in ADATOM_CELL_NAMES}
atoms_by_element = {element: Material.create(from_ase(Atoms(element, cell=[ATOM_BOX] * 3, pbc=True)))
                    for element in adatom_element_by_cell_name.values()}
for element, atom in atoms_by_element.items():
    atom.name = ATOM_NAME.format(element=element)
materials_by_name = {**slabs, **{atom.name: atom for atom in atoms_by_element.values()}}

for name, material in materials_by_name.items():
    a, b, c = material.lattice.a, material.lattice.b, material.lattice.c
    composition = "".join(f"{e}{n}" for e, n in sorted(Counter(material.basis.elements.values).items()))
    print(f"{name}: {composition}, {material.basis.number_of_atoms} atoms, "
          f"cell {a:.2f} x {b:.2f} x {c:.2f} Å")

### 3.2. Save the materials to the platform

Each adatom cell is saved with the manuscript's constraint: the adatom relaxes along z only, the C atoms in all
directions.

In [ ]:
from mat3ra.code.array_with_ids import ArrayWithIds
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_materials = {name: get_or_create_material(client, material, ACCOUNT_ID)
                   for name, material in materials_by_name.items()}
for name in ADATOM_CELL_NAMES:
    slabs[name].basis.constraints = ArrayWithIds.from_values(
        [ADATOM_CONSTRAINT if element == adatom_element_by_cell_name[name] else CARBON_CONSTRAINT
         for element in slabs[name].basis.elements.values])
    client.materials.update(saved_materials[name]["_id"], slabs[name].to_dict())

## 4. Configure the shared DFT model

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.standata.model_tree import ModelTreeStandata
from mat3ra.ade.application import Application
from mat3ra.mode import ModelFactory

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)

model_config = ModelTreeStandata.get_model_by_parameters(type="dft", subtype="gga", functional=FUNCTIONAL)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = ModelFactory.create(model_config)
print(f"Using application: {app.name}, model: {MODEL_TAG}")

## 5. Create the compute configuration
### 5.1. Select cluster

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create the compute configuration for the jobs

In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
    if cluster is None:
        raise ValueError(f"Cluster '{CLUSTER_NAME}' not found. Available: {[c['hostname'] for c in clusters]}")
else:
    cluster = clusters[0]
compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN, timeLimit=TIME_LIMIT)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}, "
      f"time limit: {TIME_LIMIT}")

## 6. Fixed-cell relaxation of the clean layer and the adatom cells

Each cell is relaxed with the dipole correction in the middle of its vacuum; the final total energy of each relaxation
is the cell energy in Eq. 1. The total magnetization of each adatom cell is read from its relaxation output. A
finished or running job for the same material and workflow name is reused, here and for the isolated atoms below.

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.workflow import apply_planewave_cutoffs, apply_scf_kgrid, patch_workflow_qe_input
from mat3ra.notebooks_utils.job import create_job
from mat3ra.notebooks_utils.core.entity.job.api import find_job_for_material

relaxation_workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(
    RELAX_WORKFLOW_SEARCH_TERM
)
relax_job_ids = {}
new_job_ids = []
for name, slab in slabs.items():
    spin_settings = ADATOM_SPIN_SETTINGS if name in ADATOM_CELL_NAMES else {}
    workflow = Workflow.create(relaxation_workflow_config)
    workflow.name = f"Relaxation {name}"
    workflow.subworkflows[0].model = model
    apply_planewave_cutoffs(workflow, ECUTWFC, ECUTRHO, unit_name=RELAX_UNIT)
    apply_scf_kgrid(workflow, SLAB_KGRID, material=slab, unit_name=RELAX_UNIT)
    patch_workflow_qe_input(workflow, {"control": RELAXATION_SETTINGS}, [RELAX_UNIT])
    patch_workflow_qe_input(workflow, DIPOLE_SETTINGS, [RELAX_UNIT])
    # emaxpos: the middle of the vacuum between the slab and its periodic image, in crystal z
    slab_z = [coordinate[2] for coordinate in slab.basis.coordinates.values]
    emaxpos = (min(slab_z) + max(slab_z) + 1) / 2 % 1
    patch_workflow_qe_input(workflow, {"system": {**SMEARING_SETTINGS, **spin_settings, "emaxpos": emaxpos}},
                            [RELAX_UNIT])
    job = find_job_for_material(client, saved_materials[name]["_id"], workflow.name, ACCOUNT_ID,
                                statuses=("submitted", "queued", "active", "finished"),
                                kgrid=SLAB_KGRID, unit_name=RELAX_UNIT)
    if job is None:
        job = create_job(
            api_client=client, materials=[saved_materials[name]], workflow=workflow, project_id=project_id,
            owner_id=ACCOUNT_ID, compute=compute.to_dict(), prefix=f"{workflow.name} {timestamp}",
        )
        new_job_ids.append(job["_id"])
        print(f"✅ {name}: created relaxation job {job['_id']}")
    else:
        print(f"♻️  {name}: reusing existing relaxation job {job['_id']}")
    relax_job_ids[name] = job["_id"]

In [ ]:
from mat3ra.notebooks_utils.api.job import submit_jobs, wait_for_jobs_to_finish_async

if new_job_ids:
    submit_jobs(client.jobs, new_job_ids)
    print(f"✅ Submitted {len(new_job_ids)} relaxation job(s).")
if relax_job_ids:
    await wait_for_jobs_to_finish_async(client.jobs, list(relax_job_ids.values()), poll_interval=POLL_INTERVAL)

In [ ]:
import re
from mat3ra.notebooks_utils.io import read_from_url

# the total magnetization of the relaxed cell is the last value in its relaxation output
total_magnetization_by_cell_name = {}
for name in ADATOM_CELL_NAMES:
    output_file = next(file for file in client.jobs.list_files(relax_job_ids[name])
                       if file["key"].endswith(RELAX_OUTPUT_FILE))
    output = await read_from_url(output_file["signedUrl"])
    magnetizations = re.findall(r"total magnetization\s+=\s+(-?\d+\.\d+) Bohr mag/cell", output)
    total_magnetization_by_cell_name[name] = abs(float(magnetizations[-1]))

## 7. Total energies
### 7.1. Isolated atoms

In [ ]:
total_energy_workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(
    TOTAL_ENERGY_SEARCH_TERM
)
atom_job_ids_by_element = {}
for element, atom in atoms_by_element.items():
    workflow = Workflow.create(total_energy_workflow_config)
    workflow.name = f"Total Energy {atom.name}"
    workflow.subworkflows[0].model = model
    apply_planewave_cutoffs(workflow, ECUTWFC, ECUTRHO, unit_name=SCF_UNIT)
    apply_scf_kgrid(workflow, GAMMA_KGRID, material=atom)
    patch_workflow_qe_input(workflow, {"system": {**SMEARING_SETTINGS, **ATOM_SPIN_SETTINGS_BY_ELEMENT[element]}},
                            [SCF_UNIT])
    job = find_job_for_material(client, saved_materials[atom.name]["_id"], workflow.name, ACCOUNT_ID,
                                statuses=("submitted", "queued", "active", "finished"), kgrid=GAMMA_KGRID)
    if job is None:
        job = create_job(
            api_client=client, materials=[saved_materials[atom.name]], workflow=workflow, project_id=project_id,
            owner_id=ACCOUNT_ID, compute=compute.to_dict(), prefix=f"{workflow.name} {timestamp}",
        )
        submit_jobs(client.jobs, [job["_id"]])
        print(f"✅ {atom.name}: created and submitted Total Energy job {job['_id']}")
    else:
        print(f"♻️  {atom.name}: reusing existing Total Energy job {job['_id']}")
    atom_job_ids_by_element[element] = job["_id"]

In [ ]:
await wait_for_jobs_to_finish_async(client.jobs, list(atom_job_ids_by_element.values()), poll_interval=POLL_INTERVAL)

### 7.2. Relaxed cells

In [ ]:
from mat3ra.notebooks_utils.core.entity.property.api import get_properties_for_job
from mat3ra.notebooks_utils.core.entity.material.api import get_final_structure_for_job

atom_energies_by_element = {element: get_properties_for_job(client, job_id, "total_energy")[0]["value"]
                            for element, job_id in atom_job_ids_by_element.items()}
relaxed_materials = {}
slab_energies = {}
print("Residual force after relaxation:")
for name, job_id in relax_job_ids.items():
    relaxed_material = get_final_structure_for_job(client, job_id)
    client.materials.update(relaxed_material.id, {"name": f"{name} relaxed"})
    relaxed_materials[name] = relaxed_material
    slab_energies[name] = get_properties_for_job(client, job_id, "total_energy")[0]["value"]
    total_force = get_properties_for_job(client, job_id, "total_force")[0]
    print(f"{name}: {total_force['value']:.4f} {total_force['units']}")

## 8. Retrieve the results
### 8.1. Adsorption energies (Eq. 1)

In [ ]:
# Eq. 1 with the sign of Table I: a bound adatom has a positive adsorption energy
adsorption_energies = {
    name: -(slab_energies[name] - atom_energies_by_element[element] - slab_energies[CLEAN_LAYER_NAME])
    for name, element in adatom_element_by_cell_name.items()}
# E_max − ΔE: how much less an adatom binds on a site than on its favored site, printed only when that cell is in
# ADATOM_CELL_NAMES. An element's favored cell is its row without an E_max − ΔE value, as Table I marks it
favored_adsorption_energies_by_element = {
    element: adsorption_energies[name] for name, element in adatom_element_by_cell_name.items()
    if "energy_below_favored_site" not in MANUSCRIPT_VALUES_BY_CELL_NAME[name]}
energies_below_favored_site = {
    name: favored_adsorption_energies_by_element[element] - adsorption_energies[name]
    for name, element in adatom_element_by_cell_name.items()
    if "energy_below_favored_site" in MANUSCRIPT_VALUES_BY_CELL_NAME[name]
    and element in favored_adsorption_energies_by_element}

### 8.2. Adatom height and nearest adatom–C distance in the relaxed cells

In [ ]:
from mat3ra.made.tools.convert import to_ase

heights = {}
adatom_carbon_distances = {}
for name in ADATOM_CELL_NAMES:
    relaxed_atoms = to_ase(relaxed_materials[name])
    symbols = relaxed_atoms.get_chemical_symbols()
    adatom_index = symbols.index(adatom_element_by_cell_name[name])
    carbon_indices = [index for index, symbol in enumerate(symbols) if symbol == "C"]
    adatom_carbon_vectors = relaxed_atoms.get_distances(adatom_index, carbon_indices, mic=True, vector=True)
    heights[name] = -adatom_carbon_vectors[:, 2].mean()
    adatom_carbon_distances[name] = min(relaxed_atoms.get_distances(adatom_index, carbon_indices, mic=True))

## 9. Comparison with Chan, Neaton & Cohen (2008)

In [ ]:
comparisons = [
    ("Adsorption energy", adsorption_energies, "adsorption_energy", "eV", 3),
    ("Adsorption energy below the favored site (E_max − ΔE)", energies_below_favored_site,
     "energy_below_favored_site", "eV", 3),
    ("Adatom height above the C plane", heights, "height", "Å", 2),
    ("Nearest adatom–C distance", adatom_carbon_distances, "adatom_carbon_distance", "Å", 2),
    ("Total magnetization", total_magnetization_by_cell_name, "total_magnetization", "μB", 2),
]
for quantity, values, manuscript_key, units, decimals in comparisons:
    manuscript_values = {name: MANUSCRIPT_VALUES_BY_CELL_NAME[name][manuscript_key] for name in values
                         if manuscript_key in MANUSCRIPT_VALUES_BY_CELL_NAME[name]}
    for header, block_values in ((quantity, values), (f"{quantity} from the manuscript", manuscript_values)):
        if block_values:
            print(f"{header}:")
            for name, value in block_values.items():
                print(f"{name}: {value:.{decimals}f} {units}")
            print()

## References

[1] Chan, K. T., Neaton, J. B., & Cohen, M. L. (2008). First-principles study of metal adatom adsorption on graphene. Phys. Rev. B, 77(23), 235430. https://doi.org/10.1103/PhysRevB.77.235430